In [1]:
# LLM prompt experiments (local dev only). Requires: poetry install --extras llm, Ollama running.
import sys
from pathlib import Path

# Make debate_analyzer importable when not run via poetry from repo root
try:
    from debate_analyzer.api.loader import load_transcript_payload
    from debate_analyzer.analysis.backend_ollama import get_ollama_backend
    from debate_analyzer.analysis.chunking import (
        DEFAULT_MAX_CHUNK_TOKENS,
        DEFAULT_OVERLAP_TOKENS,
        estimate_tokens,
        flatten_transcription,
        split_into_chunks,
    )
    from debate_analyzer.analysis.prompts import SYSTEM_PROMPT_RESPONSE_LANGUAGE
except ImportError:
    _root = Path.cwd() if (Path.cwd() / "src" / "debate_analyzer").exists() else Path.cwd().parent
    sys.path.insert(0, str(_root / "src"))
    from debate_analyzer.api.loader import load_transcript_payload
    from debate_analyzer.analysis.backend_ollama import get_ollama_backend
    from debate_analyzer.analysis.chunking import (
        DEFAULT_MAX_CHUNK_TOKENS,
        DEFAULT_OVERLAP_TOKENS,
        estimate_tokens,
        flatten_transcription,
        split_into_chunks,
    )
    from debate_analyzer.analysis.prompts import SYSTEM_PROMPT_RESPONSE_LANGUAGE

In [2]:
TRANSCRIPT_PATH = "/Users/tjirsik/Repository/debate_analyzer/data/transcripts/24.  zasedání Zastupitelstva ÚMČ Brno-Líšeň_54pamkkj3WI_transcription.json"
payload = load_transcript_payload(TRANSCRIPT_PATH)
print(f"Loaded {len(payload.get('transcription', []))} segments.")

Loaded 229 segments.


In [3]:
# Chunking: same defaults as analysis pipeline; change to experiment.
max_chunk_tokens = DEFAULT_MAX_CHUNK_TOKENS  # 24_000
overlap_tokens = DEFAULT_OVERLAP_TOKENS  # 500
chunk_index = 0  # which chunk to send to the prompt (0 = first)

In [4]:
# Build excerpt using project chunking (flatten_transcription + split_into_chunks).
flat_text = flatten_transcription(payload.get("transcription", []))
chunks = split_into_chunks(
    flat_text,
    max_tokens=max_chunk_tokens,
    overlap_tokens=overlap_tokens,
)
idx = min(chunk_index, len(chunks) - 1) if chunks else 0
if chunk_index != idx:
    print(f"chunk_index {chunk_index} out of range (0..{len(chunks) - 1}), using {idx}.")
transcript_excerpt = chunks[idx] if chunks else ""
print(f"Chunks: {len(chunks)}, selected chunk {idx}, ~{estimate_tokens(transcript_excerpt)} tokens.")
transcript_excerpt

Chunks: 3, selected chunk 0, ~23889 tokens.


'SPEAKER_15: Tak, dámy a pánové, přítomnost v sále je 16 zastupitelů, což je mírná většina nad poloviční. Co se týká programu, jak byl zaslán, já bych si dovolil proceduálně navrhnout mezi bot 1 a 2, bot 1A, kde bych, jak jsem avizoval již v mailu, chtěl navrhnout dobu jednání, a to konkrétně na 2,5 hodin, protože zde je od 18. hodiny další doba plánovaná debata. Proto tedy navrhuji doplnění bodu 1A, a to konkrétně uslyšení schválení jednací doby zasadení zastupitele z tohoto městské části od 15. do 17:30. A pak navrhuji doplnit v bodě Dum zdraví Náměstíka d. 4.\nSPEAKER_15: Představení studie Polifunčního domu Náměstíka d. 4., které by přítomní pán inž. Kaniek, architekt Koudelka prezentovali, myslím si, že by bylo dobré vědět, o čem se bavíme. To jsou návrhy, které tedy procesně bych chtěl dát s tím, že obdržíme 1A, jako v minulosti bychom hlasovali samostatně, to znamená ten časový rámec, tím, že opravdu je potřeba aspoň ta půl hodina, aby se sál uklidil a mohla tu proběhnout a vyzv

In [5]:
prompt = f"""Analyze the following meeting transcript excerpt and list the main topics discussed.

Transcript:
---
{transcript_excerpt}
---

Main topics (short list in Czech):"""

In [6]:
USE_MOCK = False  # Set True to skip Ollama and use mock responses
if USE_MOCK:
    from debate_analyzer.analysis.backend import MockLLMBackend
    backend = MockLLMBackend()
else:
    backend = get_ollama_backend(system_prompt=SYSTEM_PROMPT_RESPONSE_LANGUAGE)
response = backend.generate(prompt)
print(response)

/Users/tjirsik/Repository/debate_analyzer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[LLM] Using Ollama backend: http://localhost:11434 model=qwen2.5:7b num_ctx=8192


Zde jsou hlavní témata diskuze podle zadaného textu:

1. Projekt důmu zdravotnického na náměstí Karla 4/17a a 17b:
   - Plánovaná rekonstrukce existující budovy
   - Možnost postavení nové budovy v této lokalitě

2. Prostory pro lékaře na náměstí Karla 4/17a a 17b:
   - Nevyhovující současné prostorové podmínky
   - Riziko odchodu lékařů z důvodu nevhodných podmínek

3. Prostory pro lékárnu na náměstí Karla 4/17a:
   - Možnost umístění lékárně v rekonstruovaném domě
   - Výhody a nevýhody postavení lékárně ve proluce

4. Financování projektů na náměstí Karla 4/17a a 17b:
   - Aktuální finanční prostředky pro projekty
   - Plánované investice do budoucích roků

5. Zemní právo a prověření lokalit:
   - Proces získání povolení k rekonstrukci nebo postavení nové budovy
